# 中证800 V56：V46 单一模型族、多训练截止日与监控验证

目的：把原来混乱的“候选模型导出”流程重构成一个清晰、可复用的模型银行 notebook。

核心假设：只复刻并导出 `model_candidate_v46_lgb_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed_q4.pkl` 这一类模型；模型结构、特征、label、LightGBM 参数、固定迭代数和 bundle 格式都保持一致，唯一变量是训练窗口，也就是每个模型自己的 `train_start` 和 `train_end`。

本 notebook 系统验证：

- 不同训练起点/截止日的同一模型族，在后续年份是否稳定。
- 监控指标是否能识别 2023 或其他弱年份的问题。
- 同一个模型在 `target_top6`、`raw_top6/top8/top10/top20/top30` 和行业约束 profile 下的表现差异。

本 notebook 只导出与 `jq_backtest_v46_legacy_unsealed_monitor.py` 兼容的 direct model bundle。回测时只需要替换 `g.model_file`。

导出文件名统一带 `v56_model_bank`、`startYYYYMMDD` 和 `cutoffYYYYMMDD`，不会覆盖或混淆现有 v46/q4 模型。


数据模块默认读固定 CSV；需要重建时把 `REBUILD_DATA` 改成 `True`，会在聚宽研究环境中重新拉取指数成分、jqfactor、轻量量价/时序特征和 `alpha_1m` 标签。


In [ ]:
import os
import gc
import pickle
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

# =========================
# Config
# =========================
DATA_PATH = "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"
OUT_DIR = "csi800_ml_v56_v46_model_bank_outputs"

# 默认读固定 CSV；只有手动打开才在 JoinQuant 研究环境里重建数据。
REBUILD_DATA = False
REBUILD_INDEX = "000906.XSHG"
REBUILD_START_DATE = "2019-01-01"
REBUILD_END_DATE = "2026-05-31"
FACTOR_CHUNK_SIZE = 20
PRICE_STOCK_CHUNK_SIZE = 160

TARGET_COL = "alpha_1m"
BENCHMARK = "000906.XSHG"
DEFAULT_TRAIN_START = "2019-01-01"
MODEL_FAMILY = "v46_lgb_direct_hybrid_l2_ff10_legacy_unsealed_q4"
BOUNDARY_POLICY = "legacy_unsealed_q4"

# 训练窗口入口：每条 spec 都可以独立控制 train_start / train_end / test_start。
# tag 只是短名字；如果你留空，后面的 normalize_train_spec 会按日期自动生成。
# 模型结构完全保持 V46 q4 direct anchor，只改变训练数据时间窗。
TRAIN_WINDOW_SPECS = [
    {"tag": "2019_2022", "train_start": "2019-01-01", "train_end": "2022-12-31", "test_start": "2023-01-01"},
    {"tag": "2019_2023", "train_start": "2019-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01"},
    {"tag": "2019_2024", "train_start": "2019-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01"},
    {"tag": "2019_2025", "train_start": "2019-01-01", "train_end": "2025-12-31", "test_start": "2026-01-01"},
    # Example: rolling/shorter-history window. Uncomment if needed.
    # {"tag": "2021_2024", "train_start": "2021-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01"},
]

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
TOP_N_CANDIDATES = 30
STOCK_NUM = 6
INDUSTRY_CAP_RATIO = 0.20

PROFILE_SPECS = [
    {"profile": "raw_top6", "kind": "raw", "n": 6},
    {"profile": "raw_top8", "kind": "raw", "n": 8},
    {"profile": "raw_top10", "kind": "raw", "n": 10},
    {"profile": "raw_top20", "kind": "raw", "n": 20},
    {"profile": "raw_top30", "kind": "raw", "n": 30},
    {"profile": "industry_top6", "kind": "industry", "n": 6},
    {"profile": "industry_top10", "kind": "industry", "n": 10},
]

MONITOR_TOP_KS = [6, 10, 20, 30]
MONITOR_BOTTOM_KS = [6, 10, 30]
EXPORT_MODELS = True
RUN_OFFLINE_EVAL = True

os.makedirs(OUT_DIR, exist_ok=True)
print("output dir:", OUT_DIR)


In [ ]:
# =========================
# Feature set and LGB params
# =========================
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio",
    "book_to_price_ratio",
    "earnings_yield",
    "sales_to_price_ratio",
    "cash_earnings_to_price_ratio",
    "earnings_to_price_ratio",
    "roe_ttm",
    "roa_ttm",
    "gross_profit_ttm",
    "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit",
    "ACCA",
    "growth",
    "net_working_capital",
    "operating_profit_per_share",
    "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share",
    "super_quick_ratio",
    "MLEV",
    "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio",
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "liquidity",
    "beta",
    "ATR6",
    "MFI14",
    "DAVOL10",
    "VOL10",
    "VMACD",
    "VOSC",
    "Skewness20",
    "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

CANDIDATE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

print("candidate feature cols:", len(CANDIDATE_COLS))

## 数据模块阅读顺序

这一段只做一件事：拿到 `df_all`。默认不重建，直接读取 `DATA_PATH`；如果你把 `REBUILD_DATA=True`，才会在聚宽研究环境里重新拉取指数成分、jqfactor、轻量量价/时序特征和 `alpha_1m` 标签。


In [ ]:
# =========================
# Basic research helpers
# =========================
# Preserve feature order while removing duplicates.
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


# RankIC is the main lightweight diagnostic for monthly cross-sectional prediction quality.
def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def calc_drawdown(series):
    s = pd.Series(series).dropna()
    if len(s) == 0:
        return np.nan
    curve = (1.0 + s).cumprod()
    return float((curve / curve.cummax() - 1.0).min())


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []
    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)
    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


# Feature selection is fitted on training data only, so OOS evaluation does not leak future correlation structure.
def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
    y = d[target_col].astype(float).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


# Diagnostic validation is only for reporting; final model still trains on all rows in train_df.
def split_diag_valid(train_df):
    months = sorted(pd.to_datetime(train_df["rebalance_date"].dropna().unique()))
    if len(months) <= 8:
        return train_df.copy(), train_df.copy()
    n_valid = max(6, int(round(len(months) * 0.20)))
    valid_months = set(months[-min(n_valid, len(months) - 1):])
    fit = train_df[~train_df["rebalance_date"].isin(valid_months)].copy()
    valid = train_df[train_df["rebalance_date"].isin(valid_months)].copy()
    if fit.empty or valid.empty:
        return train_df.copy(), train_df.copy()
    return fit, valid


### 1. CSV 加载与月度调仓日历

`load_dataset` 是正常路径；`build_rebalance_schedule` 只在重建数据时使用。特征日统一取调仓日前一个交易日，避免当天不可得数据。


In [ ]:
# Load a previously built CSV. This is the default path for normal model export.
def load_dataset(path):
    if not os.path.exists(path):
        raise IOError("DATA_PATH not found: " + path + "; set REBUILD_DATA=True in JoinQuant research env to rebuild it")
    df = pd.read_csv(path)
    if "code" in df.columns and "stock" not in df.columns:
        df = df.rename(columns={"code": "stock"})
    for col in ["rebalance_date", "feature_date", "next_date"]:
        if col not in df.columns:
            raise ValueError("missing date column: " + col)
        df[col] = pd.to_datetime(df[col], errors="coerce").dt.normalize()
    if "stock" not in df.columns:
        raise ValueError("missing stock column")
    if TARGET_COL not in df.columns:
        raise ValueError("missing target column: " + TARGET_COL)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=["stock", "rebalance_date", "feature_date", "next_date", TARGET_COL]).copy()
    if "industry_bucket" not in df.columns:
        df["industry_bucket"] = "UNKNOWN"
    return df


# Rebuild schedule: monthly rebalance day, previous trading day as feature date, next rebalance as label end.
def build_rebalance_schedule():
    start_dt = pd.Timestamp(REBUILD_START_DATE)
    end_dt = pd.Timestamp(REBUILD_END_DATE)
    try:
        trade_days = pd.to_datetime(get_trade_days(start_date=(start_dt - pd.Timedelta(days=80)).date(), end_date=end_dt.date()))
    except NameError:
        raise RuntimeError("REBUILD_DATA=True requires JoinQuant get_trade_days API")
    trade_days = pd.Series(trade_days).dropna().sort_values().reset_index(drop=True)
    if trade_days.empty:
        raise ValueError("no trade days returned for rebuild window")
    month_first = []
    for _, gdf in trade_days.groupby(trade_days.dt.strftime("%Y-%m")):
        month_first.append(gdf.min())
    month_first = pd.Series(month_first).sort_values().reset_index(drop=True)
    rows = []
    all_days = pd.DatetimeIndex(trade_days)
    for i in range(len(month_first) - 1):
        rebalance_date = pd.Timestamp(month_first.iloc[i]).normalize()
        next_date = pd.Timestamp(month_first.iloc[i + 1]).normalize()
        if rebalance_date < start_dt or next_date > end_dt:
            continue
        pos = all_days.searchsorted(rebalance_date, side="left") - 1
        if pos < 0:
            continue
        rows.append({
            "rebalance_date": rebalance_date,
            "feature_date": pd.Timestamp(all_days[pos]).normalize(),
            "next_date": next_date,
        })
    return pd.DataFrame(rows)


### 2. 聚宽数据抓取函数

这些函数只服务于 `REBUILD_DATA=True`。为了和回测文件对齐，只重建 V46 当前用到的 jqfactor 特征、4 个轻量量价特征、2 个时序 rank 特征，以及 `alpha_1m` 标签。


In [ ]:
# Fetch jqfactor snapshots in small chunks to stay under JoinQuant factor API limits.
def fetch_jq_factor_snapshot(stock_list, factor_cols, date):
    out = pd.DataFrame(index=stock_list)
    if len(stock_list) == 0 or len(factor_cols) == 0:
        return out
    date_str = pd.Timestamp(date).strftime("%Y-%m-%d")
    for factor_chunk in chunks(factor_cols, FACTOR_CHUNK_SIZE):
        try:
            factor_data = get_factor_values(stock_list, factor_chunk, end_date=date_str, count=1)
        except NameError:
            raise RuntimeError("REBUILD_DATA=True requires jqfactor.get_factor_values API")
        except Exception as err:
            print("factor chunk failed", date_str, factor_chunk, err)
            factor_data = None
        for factor in factor_chunk:
            try:
                if factor_data is not None and factor in factor_data:
                    out[factor] = factor_data[factor].iloc[0, :].reindex(stock_list)
                else:
                    one = get_factor_values(stock_list, [factor], end_date=date_str, count=1)
                    out[factor] = one[factor].iloc[0, :].reindex(stock_list) if one is not None and factor in one else np.nan
            except Exception as err:
                print("factor failed", date_str, factor, err)
                out[factor] = np.nan
    return out.reindex(index=stock_list, columns=factor_cols)


def calc_ret(close_mat, days):
    if close_mat is None or close_mat.empty or len(close_mat) <= days:
        return pd.Series(dtype=float)
    return close_mat.iloc[-1] / close_mat.iloc[-days - 1] - 1


# Recreate only the four V46 lightweight price/liquidity features used by this model family.
def fetch_hybrid_price_features(stock_list, date):
    price_cols = ["liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60"]
    out = pd.DataFrame(index=stock_list, columns=price_cols, dtype=float)
    if len(stock_list) == 0:
        return out
    date_str = pd.Timestamp(date).strftime("%Y-%m-%d")
    for stock_chunk in chunks(stock_list, PRICE_STOCK_CHUNK_SIZE):
        try:
            price_df = get_price(
                stock_chunk,
                end_date=date_str,
                frequency="daily",
                fields=["close", "high", "low", "volume", "money", "paused"],
                count=121,
                skip_paused=False,
                fq="pre",
                panel=False,
                fill_paused=True,
            )
        except NameError:
            raise RuntimeError("REBUILD_DATA=True requires JoinQuant get_price API")
        except Exception as err:
            print("price feature failed", date_str, err)
            continue
        if price_df is None or price_df.empty:
            continue
        for col in ["close", "high", "low", "volume", "money", "paused"]:
            if col not in price_df.columns:
                price_df[col] = np.nan
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        money_mat = price_df.pivot_table(index="time", columns="code", values="money").sort_index()
        paused_mat = price_df.pivot_table(index="time", columns="code", values="paused").sort_index()
        if close_mat.empty:
            continue
        last_close = close_mat.iloc[-1]
        ma60 = close_mat.tail(60).mean()
        money20 = money_mat.tail(20).mean()
        money60 = money_mat.tail(60).mean()
        chunk_out = pd.DataFrame(index=stock_chunk, columns=price_cols, dtype=float)
        chunk_out["liq_money_ratio_20_60"] = money20 / money60 - 1
        chunk_out["liq_paused_count_20"] = paused_mat.tail(20).fillna(0).sum()
        chunk_out["px_close_to_ma60"] = last_close / ma60 - 1
        chunk_out["px_drawdown_60"] = last_close / close_mat.tail(60).max() - 1
        out.loc[chunk_out.index, chunk_out.columns] = chunk_out.replace([np.inf, -np.inf], np.nan)
        del price_df, close_mat, money_mat, paused_mat, chunk_out
        gc.collect()
    return out.replace([np.inf, -np.inf], np.nan).reindex(index=stock_list, columns=price_cols)


def get_month_end_feature_dates(date, months=5):
    try:
        trade_days = pd.to_datetime(get_trade_days(end_date=pd.Timestamp(date).date(), count=150))
    except NameError:
        raise RuntimeError("REBUILD_DATA=True requires JoinQuant get_trade_days API")
    if len(trade_days) == 0:
        return []
    td = pd.Series(trade_days).dropna().sort_values()
    month_last = []
    for _, gdf in td.groupby(td.dt.strftime("%Y-%m")):
        month_last.append(gdf.max())
    dates = [pd.Timestamp(d).normalize() for d in month_last if pd.Timestamp(d) <= pd.Timestamp(date)]
    return dates[-months:]


def get_factor_rank_on_date(stock_list, factor, date):
    out = pd.Series(index=stock_list, dtype=float)
    date_str = pd.Timestamp(date).strftime("%Y-%m-%d")
    try:
        factor_data = get_factor_values(stock_list, [factor], end_date=date_str, count=1)
    except NameError:
        raise RuntimeError("REBUILD_DATA=True requires jqfactor.get_factor_values API")
    except Exception as err:
        print("temporal factor failed", date_str, factor, err)
        return out
    if factor_data is None or factor not in factor_data:
        return out
    try:
        return factor_data[factor].iloc[0, :].reindex(stock_list).rank(pct=True)
    except Exception as err:
        print("temporal factor parse failed", date_str, factor, err)
        return out


# Recreate the two V46 temporal rank features using month-end factor ranks.
def fetch_temporal_features(stock_list, date):
    temporal_cols = ["ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m"]
    out = pd.DataFrame(index=stock_list, columns=temporal_cols, dtype=float)
    if len(stock_list) == 0:
        return out
    feature_dates = get_month_end_feature_dates(date, months=5)
    if len(feature_dates) < 2:
        return out.reindex(index=stock_list, columns=temporal_cols)
    cf_ranks = []
    rank1m_ranks = []
    for dt in feature_dates:
        cf_ranks.append(get_factor_rank_on_date(stock_list, "cash_flow_to_price_ratio", dt))
        rank1m_ranks.append(get_factor_rank_on_date(stock_list, "Rank1M", dt))
    if len(cf_ranks) >= 4:
        out["ts_cash_flow_to_price_ratio_rank_mean_3m"] = pd.concat(cf_ranks[-4:-1], axis=1).mean(axis=1)
    elif len(cf_ranks) >= 2:
        out["ts_cash_flow_to_price_ratio_rank_mean_3m"] = pd.concat(cf_ranks[:-1], axis=1).mean(axis=1)
    if len(rank1m_ranks) >= 2:
        out["ts_Rank1M_rank_chg_1m"] = rank1m_ranks[-1] - rank1m_ranks[-2]
    return out.replace([np.inf, -np.inf], np.nan).reindex(index=stock_list, columns=temporal_cols)


def price_series_from_get_price(price_df):
    if price_df is None or price_df.empty or "close" not in price_df.columns:
        return pd.Series(dtype=float)
    if "time" in price_df.columns:
        return pd.Series(price_df["close"].values, index=pd.to_datetime(price_df["time"]).dt.normalize()).sort_index()
    return pd.Series(price_df["close"].values).dropna()


# Label: next-month stock return minus CSI800 benchmark return.
def fetch_month_labels(stock_list, rebalance_date, next_date):
    out = pd.DataFrame(index=stock_list)
    rb = pd.Timestamp(rebalance_date).strftime("%Y-%m-%d")
    nx = pd.Timestamp(next_date).strftime("%Y-%m-%d")
    try:
        price_df = get_price(stock_list, start_date=rb, end_date=nx, frequency="daily", fields=["close"], panel=False, fq="pre", skip_paused=False, fill_paused=True)
        bench_df = get_price(BENCHMARK, start_date=rb, end_date=nx, frequency="daily", fields=["close"], panel=False, fq="pre", skip_paused=False, fill_paused=True)
    except NameError:
        raise RuntimeError("REBUILD_DATA=True requires JoinQuant get_price API")
    if price_df is None or price_df.empty:
        out["raw_return_1m"] = np.nan
    else:
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        if close_mat.empty or len(close_mat) < 2:
            out["raw_return_1m"] = np.nan
        else:
            out["raw_return_1m"] = close_mat.iloc[-1].reindex(stock_list) / close_mat.iloc[0].reindex(stock_list) - 1
    bench_s = price_series_from_get_price(bench_df)
    if len(bench_s) >= 2 and bench_s.iloc[0] != 0:
        bench_ret = float(bench_s.iloc[-1] / bench_s.iloc[0] - 1)
    else:
        bench_ret = np.nan
    out["benchmark_csi800_1m"] = bench_ret
    out[TARGET_COL] = out["raw_return_1m"] - bench_ret
    return out.replace([np.inf, -np.inf], np.nan)


# Industry bucket is only used for offline industry-constrained profile diagnostics.
def fetch_industry_bucket(stock_list, date):
    date_str = pd.Timestamp(date).strftime("%Y-%m-%d")
    out = {}
    try:
        industry_info = get_industry(stock_list, date=date_str)
    except NameError:
        print("get_industry API missing; industry_bucket will be UNKNOWN")
        return {s: "UNKNOWN" for s in stock_list}
    except Exception as err:
        print("get_industry failed", date_str, err)
        return {s: "UNKNOWN" for s in stock_list}
    for stock in stock_list:
        info = industry_info.get(stock, {}) if isinstance(industry_info, dict) else {}
        bucket = None
        for key in ["sw_l1", "jq_l1", "zjw"]:
            sub = info.get(key, None) if isinstance(info, dict) else None
            if isinstance(sub, dict):
                bucket = sub.get("industry_code") or sub.get("industry_name")
                if bucket:
                    break
        out[stock] = bucket if bucket else "UNKNOWN"
    return out


### 3. 数据重建编排与 train/test 切分

`rebuild_dataset` 按月循环生成样本；`make_train_df` 保持 `legacy_unsealed_q4` 规则，也就是按每条 spec 的 `train_start <= rebalance_date <= train_end` 截断，复刻当前强 v46 模型的训练边界，同时允许灵活改训练起点。


In [ ]:
# Full data rebuild. Use only in JoinQuant research env when REBUILD_DATA=True.
def rebuild_dataset():
    schedule = build_rebalance_schedule()
    print("rebuild schedule months:", len(schedule))
    if schedule.empty:
        raise ValueError("empty rebuild schedule")
    parts = []
    for _, row in schedule.iterrows():
        feature_date = pd.Timestamp(row["feature_date"])
        rebalance_date = pd.Timestamp(row["rebalance_date"])
        next_date = pd.Timestamp(row["next_date"])
        try:
            stock_list = list(get_index_stocks(REBUILD_INDEX, feature_date.strftime("%Y-%m-%d")))
        except NameError:
            raise RuntimeError("REBUILD_DATA=True requires JoinQuant get_index_stocks API")
        if len(stock_list) == 0:
            print("skip empty universe", rebalance_date.date())
            continue
        print("rebuild month", rebalance_date.date(), "feature", feature_date.date(), "next", next_date.date(), "stocks", len(stock_list))
        fac = fetch_jq_factor_snapshot(stock_list, BASE_FACTOR_COLS, feature_date)
        px = fetch_hybrid_price_features(stock_list, feature_date)
        ts = fetch_temporal_features(stock_list, feature_date)
        labels = fetch_month_labels(stock_list, rebalance_date, next_date)
        industry_map = fetch_industry_bucket(stock_list, feature_date)
        month_df = fac.join(px, how="left").join(ts, how="left").join(labels, how="left")
        month_df["stock"] = month_df.index
        month_df["rebalance_date"] = rebalance_date
        month_df["feature_date"] = feature_date
        month_df["next_date"] = next_date
        month_df["industry_bucket"] = pd.Series(industry_map).reindex(month_df.index).fillna("UNKNOWN").astype(str)
        parts.append(month_df.reset_index(drop=True))
        del fac, px, ts, labels, month_df
        gc.collect()
    if len(parts) == 0:
        raise ValueError("rebuild produced no rows")
    out = pd.concat(parts, ignore_index=True)
    ordered = ["stock", "rebalance_date", "feature_date", "next_date", "industry_bucket"] + CANDIDATE_COLS + ["raw_return_1m", "benchmark_csi800_1m", TARGET_COL]
    ordered = [c for c in ordered if c in out.columns]
    return out.reindex(columns=ordered)


# Single entry point: read CSV by default, rebuild and overwrite CSV only when requested.
def load_or_rebuild_dataset(path):
    if REBUILD_DATA:
        df = rebuild_dataset()
        df.to_csv(path, index=False)
        print("rebuilt data saved:", path, df.shape)
        return load_dataset(path)
    return load_dataset(path)


def normalize_train_spec(spec):
    out = dict(spec)
    out["train_start"] = out.get("train_start", DEFAULT_TRAIN_START)
    if "train_end" not in out or "test_start" not in out:
        raise ValueError("each TRAIN_WINDOW_SPECS item needs train_end and test_start")
    if not out.get("tag"):
        start_tag = pd.Timestamp(out["train_start"]).strftime("%Y%m%d")
        end_tag = pd.Timestamp(out["train_end"]).strftime("%Y%m%d")
        out["tag"] = "{}_{}".format(start_tag, end_tag)
    return out


# Legacy-unsealed cutoff: match the existing V46 q4 anchor, but let each spec set its own train_start.
def make_train_df(df_all, spec):
    spec = normalize_train_spec(spec)
    train_start = pd.Timestamp(spec["train_start"])
    train_end = pd.Timestamp(spec["train_end"])
    # Match the V46 q4 anchor: legacy_unsealed trains by rebalance_date boundaries.
    return df_all[(df_all["rebalance_date"] >= train_start) & (df_all["rebalance_date"] <= train_end)].copy()


def make_test_df(df_all, spec):
    spec = normalize_train_spec(spec)
    test_start = pd.Timestamp(spec["test_start"])
    return df_all[df_all["rebalance_date"] >= test_start].copy()


### 4. 加载数据并做基础检查

这里会打印样本规模、日期范围、目标列分布和可用特征数。跑模型前先看这几行，能快速发现路径、日期或字段不对。


In [ ]:
# Load or rebuild the dataset before model training.
df_all = load_or_rebuild_dataset(DATA_PATH)
print("loaded:", df_all.shape)
print(df_all[["rebalance_date", "feature_date", "next_date"]].agg(["min", "max"]))
print("target:", TARGET_COL)
print(df_all[[TARGET_COL]].describe())
print("available features:", len([c for c in CANDIDATE_COLS if c in df_all.columns]), "/", len(CANDIDATE_COLS))


## 模型训练与 pkl 导出

这一段固定 V46 direct LGB 训练 recipe，不做结构变化。`train_one_spec` 每次只训练一个训练窗口模型；`export_bundle` 保持和回测文件兼容。


In [ ]:
# =========================
# Train, score, export
# This cell keeps the model recipe fixed; only train_start/train_end changes across exported pkl files.
# =========================
def train_direct_lgb(train_df, feature_cols):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, train_index = prepare_xy(train_df, feature_cols, TARGET_COL)
    model = lgb.train(
        params,
        lgb.Dataset(X_train, label=y_train),
        num_boost_round=max(1, int(FIXED_ITER)),
    )
    pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    train_rank_ic = safe_rank_ic(y_train, pred)
    return {
        "model": model,
        "fill_values": fill_values,
        "train_rows": int(len(X_train)),
        "train_rank_ic": train_rank_ic,
    }


def score_with_model(df, model, feature_cols, fill_values):
    X = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(fill_values).fillna(0)
    return np.asarray(model.predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)


# Include v56_model_bank, start date and cutoff date so exported pkl names never collide with existing anchors.
def make_model_id(spec):
    spec = normalize_train_spec(spec)
    train_start_tag = pd.Timestamp(spec["train_start"]).strftime("%Y%m%d")
    train_end_tag = pd.Timestamp(spec["train_end"]).strftime("%Y%m%d")
    return "v56_model_bank_{}_{}_start{}_cutoff{}_fixed{}".format(
        MODEL_FAMILY,
        spec["tag"],
        train_start_tag,
        train_end_tag,
        int(FIXED_ITER),
    )


# Bundle schema matches jq_backtest_v46_legacy_unsealed_monitor.py.
def export_bundle(model_id, spec, trained, feature_cols, removed_cols, diag_rank_ic, train_df):
    model_file = "model_candidate_{}.pkl".format(model_id)
    out_path = os.path.join(OUT_DIR, model_file)
    bundle = {
        "objective": "v210_refit_fixed_iter_overlay",
        "research_version": model_id,
        "benchmark": BENCHMARK,
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "label_end": spec["train_end"],
        "boundary_policy": BOUNDARY_POLICY,
        "require_label_end_within_train": False,
        "model_family": MODEL_FAMILY,
        "target_col": TARGET_COL,
        "target_note": "V56 V46 q4 model bank: direct LGB alpha_1m, fixed iteration, no early stopping",
        "data_file": DATA_PATH,
        "protocol": "v56_v46_q4_single_family_fixed_cutoff",
        "training_policy": "fixed_window_legacy_unsealed",
        "param_set": "v46_base_ff10_original",
        "base_params": dict(BASE_PARAMS_FF10),
        "base_model": trained["model"],
        "base_feature_cols": list(feature_cols),
        "base_fill_values": dict(trained["fill_values"]),
        "base_best_iter": int(FIXED_ITER),
        "model_iter": int(FIXED_ITER),
        "fixed_iter": int(FIXED_ITER),
        "base_inner_metrics": {
            "train_rank_ic": float(trained["train_rank_ic"]) if not pd.isnull(trained["train_rank_ic"]) else np.nan,
            "diag_rank_ic": float(diag_rank_ic) if not pd.isnull(diag_rank_ic) else np.nan,
        },
        "base_removed_features": list(removed_cols),
        "residual_model": None,
        "residual_feature_cols": [],
        "residual_fill_values": {},
        "overlay_weight": 0.0,
        "overlay_mode": "direct",
        "top_n_candidates": TOP_N_CANDIDATES,
        "stock_num": STOCK_NUM,
        "industry_cap_ratio": INDUSTRY_CAP_RATIO,
        "requires_v4_feature_adapter": True,
        "requires_industry_relative_adapter": False,
        "uses_time_weight": False,
        "uses_sample_weight": False,
        "uses_current_valid_for_training": False,
        "final_role": "v46_lgb_direct_hybrid_l2_ff10_legacy_unsealed_q4_cutoff_anchor",
        "train_row_count": int(len(train_df)),
        "train_month_count": int(train_df["rebalance_date"].nunique()),
        "max_train_rebalance_date": str(train_df["rebalance_date"].max().date()),
        "max_train_next_date": str(train_df["next_date"].max().date()),
    }
    if EXPORT_MODELS:
        with open(out_path, "wb") as f:
            pickle.dump(bundle, f, protocol=2)
    return out_path, model_file


# Train exactly one V46-style direct LGB model for one cutoff spec.
def train_one_spec(df_all, spec):
    spec = normalize_train_spec(spec)
    train_df = make_train_df(df_all, spec)
    if train_df.empty:
        raise ValueError("empty train_df for " + str(spec["tag"]))
    diag_fit_df, diag_valid_df = split_diag_valid(train_df)
    feature_cols, removed_cols = select_features_train_only(diag_fit_df, CANDIDATE_COLS)
    trained = train_direct_lgb(train_df, feature_cols)
    X_valid, y_valid, _, _ = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, trained["fill_values"])
    valid_pred = np.asarray(trained["model"].predict(X_valid[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    diag_rank_ic = safe_rank_ic(y_valid, valid_pred)
    model_id = make_model_id(spec)
    model_path, model_file = export_bundle(model_id, spec, trained, feature_cols, removed_cols, diag_rank_ic, train_df)
    row = {
        "model_id": model_id,
        "model_family": MODEL_FAMILY,
        "boundary_policy": BOUNDARY_POLICY,
        "tag": spec["tag"],
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "train_rows": int(len(train_df)),
        "train_months": int(train_df["rebalance_date"].nunique()),
        "max_train_next_date": str(train_df["next_date"].max().date()),
        "feature_count": len(feature_cols),
        "removed_feature_count": len(removed_cols),
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": diag_rank_ic,
        "model_file": model_file,
        "model_path": model_path,
    }
    return row, trained, feature_cols


## 离线 profile 与监控回放

这里不是训练多个模型，而是用同一个模型分数评估不同组合口径：raw topN、行业约束 topN，以及和回测日志一致的 TopK/BottomK/rank 监控指标。


In [ ]:
# =========================
# Offline profiles and monitor replay
# Profiles are diagnostics for the same model score, not different exported models.
# =========================
def build_industry_neutral_targets(sorted_stocks, industry_map, target_num, max_per_industry):
    known_industries = set([
        industry_map.get(stock, "UNKNOWN")
        for stock in sorted_stocks
        if industry_map.get(stock, "UNKNOWN") != "UNKNOWN"
    ])
    if len(known_industries) < 3:
        return sorted_stocks[:min(target_num, len(sorted_stocks))]

    selected = []
    industry_count = {}
    for stock in sorted_stocks:
        industry = industry_map.get(stock, "UNKNOWN")
        if industry == "UNKNOWN":
            continue
        if industry_count.get(industry, 0) == 0:
            selected.append(stock)
            industry_count[industry] = 1
            if len(selected) >= target_num:
                return selected

    for stock in sorted_stocks:
        if stock in selected:
            continue
        industry = industry_map.get(stock, "UNKNOWN")
        if industry == "UNKNOWN":
            continue
        cnt = industry_count.get(industry, 0)
        if cnt < max_per_industry:
            selected.append(stock)
            industry_count[industry] = cnt + 1
            if len(selected) >= target_num:
                return selected

    for stock in sorted_stocks:
        if stock not in selected:
            selected.append(stock)
            if len(selected) >= target_num:
                break
    return selected


def select_profile_stocks(month_df, profile_spec):
    sorted_stocks = list(month_df.sort_values("score", ascending=False)["stock"])
    n = int(profile_spec["n"])
    if profile_spec["kind"] == "raw":
        return sorted_stocks[:min(n, len(sorted_stocks))]
    industry_map = dict(zip(month_df["stock"], month_df["industry_bucket"].fillna("UNKNOWN").astype(str)))
    return build_industry_neutral_targets(
        sorted_stocks,
        industry_map,
        n,
        max(1, int(np.floor(n * INDUSTRY_CAP_RATIO))),
    )


# These monitor stats mirror the readable logs added to the JoinQuant backtest file.
def monitor_stats_for_selection(month_df, selected):
    ret = month_df.set_index("stock")[TARGET_COL].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    selected = [s for s in selected if s in ret.index]
    result = {
        "valid_count": len(selected),
        "avg_ret": np.nan,
        "median_ret": np.nan,
        "worst_ret": np.nan,
        "avg_rank": np.nan,
        "median_rank": np.nan,
        "worst_rank": np.nan,
    }
    if len(ret) == 0 or len(selected) == 0:
        return result
    selected_ret = ret.reindex(selected).dropna()
    rank_pct = ret.rank(pct=True).reindex(selected_ret.index).dropna()
    if len(selected_ret) > 0:
        result["avg_ret"] = float(selected_ret.mean())
        result["median_ret"] = float(selected_ret.median())
        result["worst_ret"] = float(selected_ret.min())
    if len(rank_pct) > 0:
        result["avg_rank"] = float(rank_pct.mean())
        result["median_rank"] = float(rank_pct.median())
        result["worst_rank"] = float(rank_pct.min())

    universe_n = len(ret)
    for k in MONITOR_TOP_KS:
        k = int(k)
        top = set(ret.sort_values(ascending=False).index[:min(k, universe_n)])
        hit = len([s for s in selected if s in top])
        expected = float(len(selected) * min(k, universe_n)) / float(max(1, universe_n))
        result["T{}_hit".format(k)] = hit
        result["T{}_lift".format(k)] = float(hit) / expected if expected > 0 else np.nan
    for k in MONITOR_BOTTOM_KS:
        k = int(k)
        bottom = set(ret.sort_values(ascending=True).index[:min(k, universe_n)])
        result["B{}".format(k)] = len([s for s in selected if s in bottom])
    return result


def evaluate_model_oos(df_all, row, trained, feature_cols):
    spec = {
        "tag": row["tag"],
        "train_start": row["train_start"],
        "train_end": row["train_end"],
        "test_start": row["test_start"],
    }
    test_df = make_test_df(df_all, spec)
    if test_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    test_df = test_df.copy()
    test_df["score"] = score_with_model(test_df, trained["model"], feature_cols, trained["fill_values"])

    monthly_rows = []
    monitor_rows = []
    latest_rows = []
    for rebalance_date, month_df in test_df.groupby("rebalance_date"):
        month_df = month_df.dropna(subset=[TARGET_COL, "score"]).copy()
        if month_df.empty:
            continue
        rank_ic = safe_rank_ic(month_df["score"], month_df[TARGET_COL])
        for profile_spec in PROFILE_SPECS:
            selected = select_profile_stocks(month_df, profile_spec)
            selected_ret = month_df.set_index("stock")[TARGET_COL].reindex(selected).dropna()
            ret = float(selected_ret.mean()) if len(selected_ret) else np.nan
            monthly_rows.append({
                "model_id": row["model_id"],
                "model_family": row["model_family"],
                "tag": row["tag"],
                "train_start": row["train_start"],
                "train_end": row["train_end"],
                "test_start": row["test_start"],
                "rebalance_date": rebalance_date,
                "profile": profile_spec["profile"],
                "target_count": len(selected),
                "alpha_mean": ret,
                "rank_ic": rank_ic,
                "targets": ",".join(selected),
            })
            stats = monitor_stats_for_selection(month_df, selected)
            stats.update({
                "model_id": row["model_id"],
                "model_family": row["model_family"],
                "tag": row["tag"],
                "train_start": row["train_start"],
                "train_end": row["train_end"],
                "test_start": row["test_start"],
                "rebalance_date": rebalance_date,
                "profile": profile_spec["profile"],
                "rank_ic": rank_ic,
            })
            monitor_rows.append(stats)

    monthly_df = pd.DataFrame(monthly_rows)
    monitor_df = pd.DataFrame(monitor_rows)
    if not monthly_df.empty:
        latest_month = monthly_df["rebalance_date"].max()
        latest_rows = monthly_df[monthly_df["rebalance_date"] == latest_month].copy()
    latest_df = pd.DataFrame(latest_rows)
    return monthly_df, monitor_df, latest_df


def summarize_monthly(monthly_df):
    rows = []
    if monthly_df.empty:
        return pd.DataFrame()
    for keys, gdf in monthly_df.groupby(["model_id", "model_family", "tag", "train_start", "train_end", "test_start", "profile"]):
        s = gdf.sort_values("rebalance_date")["alpha_mean"].replace([np.inf, -np.inf], np.nan).dropna()
        ric = gdf["rank_ic"].replace([np.inf, -np.inf], np.nan).dropna()
        if len(s) == 0:
            continue
        rows.append({
            "model_id": keys[0],
            "model_family": keys[1],
            "tag": keys[2],
            "train_start": keys[3],
            "train_end": keys[4],
            "test_start": keys[5],
            "profile": keys[6],
            "months": int(len(s)),
            "cum_alpha": float((1.0 + s).prod() - 1.0),
            "mean_alpha": float(s.mean()),
            "win_rate": float((s > 0).mean()),
            "max_drawdown": calc_drawdown(s),
            "rank_ic_mean": float(ric.mean()) if len(ric) else np.nan,
            "rank_ic_ir": float(ric.mean() / ric.std()) if len(ric) > 1 and ric.std() > 0 else np.nan,
            "drop_top1_cum_alpha": float((1.0 + s.drop(s.idxmax())).prod() - 1.0) if len(s) > 1 else np.nan,
        })
    return pd.DataFrame(rows).sort_values(["profile", "train_start", "train_end", "cum_alpha"], ascending=[True, True, True, False])


## 批量训练不同截止日模型

循环 `TRAIN_WINDOW_SPECS`，导出多个不同训练起止日期的同模型族 pkl，并保存 manifest/monthly/monitor/summary/latest targets。


In [ ]:
# =========================
# Run model bank
# =========================
manifest_rows = []
monthly_parts = []
monitor_parts = []
latest_parts = []

for spec in TRAIN_WINDOW_SPECS:
    print("")
    spec = normalize_train_spec(spec)
    print("=== train", MODEL_FAMILY, spec["tag"], spec["train_start"], "to", spec["train_end"], "===")
    row, trained, feature_cols = train_one_spec(df_all, spec)
    manifest_rows.append(row)
    print("  rows:", row["train_rows"], "months:", row["train_months"], "features:", row["feature_count"])
    print("  train_rank_ic:", row["train_rank_ic"], "diag_rank_ic:", row["diag_rank_ic"])
    print("  model:", row["model_path"])

    if RUN_OFFLINE_EVAL:
        monthly_df, monitor_df, latest_df = evaluate_model_oos(df_all, row, trained, feature_cols)
        if not monthly_df.empty:
            monthly_parts.append(monthly_df)
        if not monitor_df.empty:
            monitor_parts.append(monitor_df)
        if not latest_df.empty:
            latest_parts.append(latest_df)
    del trained
    gc.collect()

manifest_df = pd.DataFrame(manifest_rows)
monthly_df = pd.concat(monthly_parts, ignore_index=True) if len(monthly_parts) else pd.DataFrame()
monitor_df = pd.concat(monitor_parts, ignore_index=True) if len(monitor_parts) else pd.DataFrame()
latest_df = pd.concat(latest_parts, ignore_index=True) if len(latest_parts) else pd.DataFrame()
summary_df = summarize_monthly(monthly_df)

manifest_path = os.path.join(OUT_DIR, "v56_model_bank_export_manifest.csv")
monthly_path = os.path.join(OUT_DIR, "v56_model_bank_monthly.csv")
monitor_path = os.path.join(OUT_DIR, "v56_model_bank_monitor.csv")
summary_path = os.path.join(OUT_DIR, "v56_model_bank_summary.csv")
latest_path = os.path.join(OUT_DIR, "v56_model_bank_latest_targets.csv")

manifest_df.to_csv(manifest_path, index=False)
monthly_df.to_csv(monthly_path, index=False)
monitor_df.to_csv(monitor_path, index=False)
summary_df.to_csv(summary_path, index=False)
latest_df.to_csv(latest_path, index=False)

print("")
print("saved:")
for p in [manifest_path, monthly_path, monitor_path, summary_path, latest_path]:
    print(" ", p)

print("")
print("manifest")
display(manifest_df)
print("")
print("summary top rows")
display(summary_df.sort_values(["profile", "cum_alpha"], ascending=[True, False]).head(80))


## 结果诊断视图

先看不同训练截止日和 profile 的 OOS 汇总，再看监控样本。如果后续回测异常，可以用这里定位是模型 score、候选池 recall，还是组合约束造成的差异。


In [ ]:
# =========================
# Diagnostics: which cutoff/profile is worth backtesting?
# =========================
if not summary_df.empty:
    focus_profiles = ["industry_top6", "raw_top6", "raw_top10", "raw_top20"]
    focus = summary_df[summary_df["profile"].isin(focus_profiles)].copy()
    print("By train cutoff / profile")
    display(focus.sort_values(["train_start", "train_end", "profile"]))

if not monitor_df.empty:
    # 监控重点：target/top6 的 rank、bottom 命中、raw_top30 的 lift 是否提前恶化。
    watch_profiles = ["industry_top6", "raw_top6", "raw_top10", "raw_top30"]
    watch = monitor_df[monitor_df["profile"].isin(watch_profiles)].copy()
    cols = [
        "model_id", "train_start", "train_end", "rebalance_date", "profile",
        "avg_ret", "avg_rank", "worst_rank", "T20_lift", "T30_lift", "B30", "rank_ic"
    ]
    cols = [c for c in cols if c in watch.columns]
    print("Monitor samples")
    display(watch[cols].sort_values(["train_start", "train_end", "rebalance_date", "profile"]).head(120))

    monitor_summary = watch.groupby(["model_id", "train_start", "train_end", "profile"]).agg(
        months=("rebalance_date", "nunique"),
        avg_ret=("avg_ret", "mean"),
        avg_rank=("avg_rank", "mean"),
        worst_rank_mean=("worst_rank", "mean"),
        t20_lift=("T20_lift", "mean"),
        t30_lift=("T30_lift", "mean"),
        b30_mean=("B30", "mean"),
        rank_ic=("rank_ic", "mean"),
    ).reset_index()
    print("Monitor summary")
    display(monitor_summary.sort_values(["profile", "train_start", "train_end"]))


## 使用建议

1. 先看 `v56_model_bank_export_manifest.csv`：确认 pkl 是否按训练窗口导出，文件名均带 `v56_model_bank`、`startYYYYMMDD` 和 `cutoffYYYYMMDD`。
2. 再看 `v56_model_bank_summary.csv`：同一 V46 q4 模型族在不同训练截止日后的 OOS 表现。
3. 再看 `v56_model_bank_monitor.csv`：判断弱年份到底是候选池失效、组合构建拖累，还是只是没抓到极端赢家。
4. 真正上传 JoinQuant 回测的文件来自 manifest 的 `model_path`。
5. 回测文件继续使用 `jq_backtest_v46_legacy_unsealed_monitor.py`，只替换 `g.model_file`。
6. raw/industry profile 只是同一模型的离线评估口径，不代表导出了不同模型策略。

7. 如需重新生成训练数据，把配置区 `REBUILD_DATA = True`；默认 `False` 会直接读取 `DATA_PATH`，不调用聚宽 API。
